# GeoMapBench — rebuild only the eight corrected tasks

This notebook uses the updated `GeoMapBench-updated.zip` codebase and regenerates only:

1. `coordinate_transformation`
2. `cross_entity_comparison`
3. `dense_land_cover_labeling`
4. `environmental_layer_identification`
5. `geo_entity_typing`
6. `isochrone_service_area`
7. `map_label_feature_anchoring`
8. `map_text_detection_recognition_grouping`

## DO NOT RUN / unchanged tasks

The other 15 task generators are unchanged and are intentionally absent from this notebook. Their existing folders in Google Drive are preserved.

Before running, place `GeoMapBench-updated.zip` in:

`MyDrive/GeoMapBench_Data/GeoMapBench-updated.zip`

If it is not there, the notebook will ask you to upload it.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/GeoMapBench_Data')
CODE_ZIP = DRIVE_ROOT / 'GeoMapBench-updated.zip'
CACHE = DRIVE_ROOT / 'cache'
DOWNLOADS = DRIVE_ROOT / 'downloads'
FINAL_OUT = DRIVE_ROOT / 'geomapbench_100'
METADATA = DRIVE_ROOT / 'release_metadata'

REPO = Path('/content/GeoMapBench')
EXTRACT_ROOT = Path('/content/geomapbench_code_extract')
RAW = Path('/content/geomapbench_raw')
LOCAL_OUT = Path('/content/geomapbench_build')

for path in [DRIVE_ROOT, CACHE, DOWNLOADS, FINAL_OUT, METADATA, RAW, LOCAL_OUT]:
    path.mkdir(parents=True, exist_ok=True)

if not CODE_ZIP.is_file():
    from google.colab import files

    print(f'Missing {CODE_ZIP}. Select the provided GeoMapBench-updated.zip file.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one GeoMapBench-updated.zip file.')
    uploaded_name = next(iter(uploaded))
    shutil.copy2(uploaded_name, CODE_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
shutil.rmtree(REPO, ignore_errors=True)
EXTRACT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.extractall(EXTRACT_ROOT)

project_files = list(EXTRACT_ROOT.rglob('pyproject.toml'))
if len(project_files) != 1:
    raise RuntimeError(f'Expected one pyproject.toml, found: {project_files}')
project_root = project_files[0].parent
shutil.copytree(project_root, REPO)

os.environ.update({
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'CACHE': str(CACHE),
    'DOWNLOADS': str(DOWNLOADS),
    'FINAL_OUT': str(FINAL_OUT),
    'REPO': str(REPO),
    'RAW': str(RAW),
    'LOCAL_OUT': str(LOCAL_OUT),
})

print('Repository:', REPO)
print('Final data root:', FINAL_OUT)

In [ ]:
%%bash
set -euo pipefail

apt-get update -qq
apt-get install -y -qq git unzip libspatialindex-dev
python -m pip install --upgrade pip setuptools wheel -q
python -m pip install -e "$REPO" requests tqdm pytest -q

python -m compileall -q "$REPO/geomapbench_data"
PYTHONPATH="$REPO" pytest -q "$REPO/tests"
geomapbench-data seeds >/dev/null

echo "Updated codebase installed and tests passed."

## Build/publish helper

Each corrected task is generated locally, validated, copied to a same-name staging folder in Drive, validated again, and only then replaces the old task folder. A failed build never deletes the existing Drive version.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

from geomapbench_data.common import DATA_REVISION
from geomapbench_data.validate import validate_task

AFFECTED = {
    "coordinate_transformation",
    "cross_entity_comparison",
    "dense_land_cover_labeling",
    "environmental_layer_identification",
    "geo_entity_typing",
    "isochrone_service_area",
    "map_label_feature_anchoring",
    "map_text_detection_recognition_grouping",
}


def run_and_publish(command: str, leaf: str, arguments: list[str]) -> None:
    if leaf not in AFFECTED:
        raise ValueError(f"Refusing to rebuild unchanged task: {leaf}")

    local_task = LOCAL_OUT / leaf
    final_task = FINAL_OUT / leaf
    staging_root = FINAL_OUT / ".incoming_revised_tasks"
    staging_task = staging_root / leaf

    shutil.rmtree(local_task, ignore_errors=True)
    shutil.rmtree(staging_task, ignore_errors=True)

    cmd = ["geomapbench-data", command, *arguments, "--output", str(LOCAL_OUT)]
    print("\nRUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

    errors = validate_task(local_task)
    if errors:
        raise RuntimeError("Local validation failed:\n" + "\n".join(errors))

    manifest = json.loads((local_task / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("data_revision") != DATA_REVISION:
        raise RuntimeError(f"{leaf}: wrong data revision {manifest.get('data_revision')}")

    staging_root.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_task, staging_task)

    errors = validate_task(staging_task)
    if errors:
        shutil.rmtree(staging_task, ignore_errors=True)
        raise RuntimeError("Drive staging validation failed:\n" + "\n".join(errors))

    backup = FINAL_OUT / f".{leaf}_previous"
    shutil.rmtree(backup, ignore_errors=True)
    if final_task.exists():
        final_task.rename(backup)
    try:
        staging_task.rename(final_task)
    except Exception:
        if backup.exists() and not final_task.exists():
            backup.rename(final_task)
        raise
    shutil.rmtree(backup, ignore_errors=True)

    print(f"COMPLETE: {leaf}")
    print("Saved to:", final_task)
    print("Revision:", manifest["data_revision"])


## Prepare MapText source

This is a small/medium official Zenodo source and is downloaded automatically if it is not cached locally.

In [ ]:
import subprocess

MAPTEXT_SOURCE = RAW / 'maptext'
subprocess.run(
    ['geomapbench-data', 'fetch', 'maptext', '--raw-root', str(RAW)],
    check=True,
)
print('MapText source:', MAPTEXT_SOURCE)

## Prepare OpenEarthMap source

The archive is about 9.1 GB. The cell reuses/resumes the persistent Drive download and extracts it into the current Colab runtime. It does not download again when the Drive archive is complete.

In [ ]:
import re
import requests
import zipfile
from tqdm.auto import tqdm

OPENEARTH_ARCHIVE = DOWNLOADS / 'OpenEarthMap.zip'
OPENEARTH_URL = 'https://zenodo.org/records/7223446/files/OpenEarthMap.zip?download=1'
OPENEARTH_EXPECTED_SIZE = 9_099_481_727
OPENEARTH_SOURCE = RAW / 'openearthmap'

current_size = OPENEARTH_ARCHIVE.stat().st_size if OPENEARTH_ARCHIVE.exists() else 0
if current_size > OPENEARTH_EXPECTED_SIZE:
    raise RuntimeError('Cached OpenEarthMap.zip is larger than the official archive.')

if current_size < OPENEARTH_EXPECTED_SIZE:
    headers = {
        'Range': f'bytes={current_size}-',
        'User-Agent': 'GeoMapBenchDataKit/1.1',
        'Accept-Encoding': 'identity',
    }
    response = requests.get(
        OPENEARTH_URL,
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(60, 600),
    )
    response.raise_for_status()
    if current_size and response.status_code != 206:
        response.close()
        raise RuntimeError(
            f'The server did not accept resume mode (HTTP {response.status_code}). '
            'The partial Drive file was not modified.'
        )
    content_range = response.headers.get('Content-Range', '')
    match = re.search(r'/(\d+)$', content_range)
    total = int(match.group(1)) if match else OPENEARTH_EXPECTED_SIZE
    mode = 'ab' if current_size else 'wb'
    with OPENEARTH_ARCHIVE.open(mode) as target, tqdm(
        total=total,
        initial=current_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
        desc='OpenEarthMap',
        dynamic_ncols=True,
    ) as progress:
        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
            if chunk:
                target.write(chunk)
                progress.update(len(chunk))
    response.close()

actual_size = OPENEARTH_ARCHIVE.stat().st_size
if actual_size != OPENEARTH_EXPECTED_SIZE:
    raise RuntimeError(
        f'OpenEarthMap archive is incomplete: {actual_size} / {OPENEARTH_EXPECTED_SIZE} bytes. '
        'Rerun this cell to resume.'
    )

marker = OPENEARTH_SOURCE / '.extracted_size'
if not marker.exists() or marker.read_text().strip() != str(actual_size):
    shutil.rmtree(OPENEARTH_SOURCE, ignore_errors=True)
    OPENEARTH_SOURCE.mkdir(parents=True)
    with zipfile.ZipFile(OPENEARTH_ARCHIVE) as archive:
        # Extraction verifies each member CRC. Avoid testzip(), which would read
        # the entire 9.1 GB archive twice.
        archive.extractall(OPENEARTH_SOURCE)
    marker.write_text(str(actual_size), encoding='utf-8')

print('OpenEarthMap source:', OPENEARTH_SOURCE)
print('Files:', sum(1 for path in OPENEARTH_SOURCE.rglob('*') if path.is_file()))

## Rebuild source-backed corrected tasks

In [ ]:
run_and_publish(
    'maptext',
    'map_text_detection_recognition_grouping',
    ['--source', str(MAPTEXT_SOURCE)],
)

In [ ]:
run_and_publish(
    'openearthmap',
    'dense_land_cover_labeling',
    ['--source', str(OPENEARTH_SOURCE)],
)

## Rebuild corrected static/API tasks

The environmental generator now downloads only the official WorldClim 10-minute elevation and bioclimatic archives; the old multi-gigabyte Köppen archive is not needed.

In [ ]:
run_and_publish(
    'coordinate-transform',
    'coordinate_transformation',
    ['--cache', str(CACHE)],
)

run_and_publish(
    'geo-entity-typing',
    'geo_entity_typing',
    ['--cache', str(CACHE)],
)

run_and_publish(
    'cross-entity-comparison',
    'cross_entity_comparison',
    ['--cache', str(CACHE)],
)

run_and_publish(
    'environmental-layer',
    'environmental_layer_identification',
    ['--cache', str(CACHE)],
)

## Rebuild corrected OpenStreetMap tasks

These cells use 20 geographically diverse cities. OSM/Overpass responses are cached in Drive. Run each cell independently so a temporary API failure can be retried without repeating the other task.

In [ ]:
run_and_publish(
    'osm-label-anchoring',
    'map_label_feature_anchoring',
    ['--cache', str(CACHE)],
)

In [ ]:
run_and_publish(
    'osm-isochrone',
    'isochrone_service_area',
    ['--cache', str(CACHE)],
)

## Validate the revised tasks and the complete benchmark

The first check validates only the eight corrected folders. The second check validates all 23 existing task folders without regenerating the unchanged ones.

In [ ]:
from geomapbench_data.validate import validate_root, validate_task

for leaf in sorted(AFFECTED):
    errors = validate_task(FINAL_OUT / leaf)
    if errors:
        raise RuntimeError(f"{leaf}:\n" + "\n".join(errors))
    print("PASS:", leaf)

all_errors = validate_root(FINAL_OUT, require_all=True, require_assets=True)
if all_errors:
    raise RuntimeError("Full benchmark validation failed:\n" + "\n".join(all_errors))

print("\nAll 23 tasks passed validation. Only the eight affected folders were regenerated.")


## Freeze release metadata and checksums

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

METADATA.mkdir(parents=True, exist_ok=True)
checksums = {}
for task_dir in sorted(path for path in FINAL_OUT.iterdir() if path.is_dir() and not path.name.startswith(".")):
    data_path = task_dir / "data.jsonl"
    manifest_path = task_dir / "manifest.json"
    if data_path.is_file() and manifest_path.is_file():
        checksums[task_dir.name] = {
            "data_jsonl_sha256": hashlib.sha256(data_path.read_bytes()).hexdigest(),
            "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
            "data_revision": json.loads(manifest_path.read_text()).get("data_revision"),
        }

release = {
    "created_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "code_version": "1.1.0",
    "revised_data_revision": DATA_REVISION,
    "regenerated_tasks": sorted(AFFECTED),
    "unchanged_tasks_were_not_regenerated": True,
    "checksums": checksums,
}
release_path = METADATA / "geomapbench_release_1_1_0.json"
release_path.write_text(json.dumps(release, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print("Release metadata:", release_path)
print("Task checksums:", len(checksums))
